# Experiment 7: percentile sweep for targeted training with a frozen FinBERT CNN

This notebook compares a frozen-FinBERT CNN baseline with targeted and matched random-control training across several hard-example percentiles. Hard examples are selected with the project's `utils.hard_samples_identify` margin utilities. Training selection uses out-of-fold predictions; validation and test subsets use the same percentile rule on baseline predictions. Results include overall and hard-subset accuracy/macro-F1 plus deltas relative to the baseline.

## Imports

In [17]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import pickle
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoModel, AutoTokenizer

sys.path.append("..")
from utils.preprocessing import LABELS, encode_labels
from utils.hard_samples_identify import get_hard_sample_indices

keras.utils.set_random_seed(42)


## Load FinBERT embeddings and tokenize

In [18]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

finbert_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(finbert_name)
finbert = AutoModel.from_pretrained(finbert_name)
embedding_matrix = finbert.embeddings.word_embeddings.weight.detach().cpu().numpy()

X_train = tokenizer(train_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_val = tokenizer(val_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_test = tokenizer(test_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]

del finbert
print(X_train.shape, X_val.shape, X_test.shape, embedding_matrix.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2901, 64) (967, 64) (968, 64) (30522, 768)


## Build the CNN

In [19]:
def build_cnn():
    inputs = keras.Input(shape=(64,), dtype="int32")
    x = layers.Embedding(
        embedding_matrix.shape[0], embedding_matrix.shape[1],
        embeddings_initializer=keras.initializers.Constant(embedding_matrix),
        trainable=False, mask_zero=False,
    )(inputs)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Conv1D(64, 3, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(LABELS), activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

class MacroF1Callback(keras.callbacks.Callback):
    def __init__(self, X, y):
        super().__init__(); self.X = X; self.y = y
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        predictions = self.model.predict(self.X, batch_size=32, verbose=0).argmax(axis=1)
        logs["val_macro_f1"] = f1_score(self.y, predictions, average="macro")
        print(f" - val_macro_f1: {logs['val_macro_f1']:.4f}")

def evaluate(model, X, y, name):
    probabilities = model.predict(X, batch_size=32, verbose=0)
    predictions = probabilities.argmax(axis=1)
    print(name)
    print("Accuracy:", accuracy_score(y, predictions))
    print("Macro F1:", f1_score(y, predictions, average="macro"))
    print(classification_report(y, predictions, target_names=LABELS))
    ConfusionMatrixDisplay.from_predictions(y, predictions, display_labels=LABELS)
    return probabilities, predictions


## Train the baseline

In [20]:
baseline_model = build_cnn()
baseline_model.fit(
    X_train, keras.utils.to_categorical(y_train, len(LABELS)),
    validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS))),
    epochs=20, batch_size=32,
    callbacks=[MacroF1Callback(X_val, y_val), keras.callbacks.EarlyStopping(
        monitor="val_macro_f1", mode="max", patience=3, restore_best_weights=True
    )],
)


Epoch 1/20
89/91 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5871 - loss: 0.9642 - val_macro_f1: 0.2688
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5891 - loss: 0.9619 - val_accuracy: 0.6019 - val_loss: 0.9032 - val_macro_f1: 0.2688
Epoch 2/20
89/91 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6173 - loss: 0.8799 - val_macro_f1: 0.3800
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6194 - loss: 0.8770 - val_accuracy: 0.6515 - val_loss: 0.8235 - val_macro_f1: 0.3800
Epoch 3/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6389 - loss: 0.8384 - val_macro_f1: 0.4066
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.6405 - loss: 0.8364 - val_accuracy: 0.6711 - val_loss: 0.7767 - val_macro_f1: 0.4066
Epoch 4/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6733 - loss: 0.7663 - val_macro_f1: 0.4404
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6736 - loss: 0.7651 - val_accuracy: 0.6887 - val_loss: 0.7279 - val_macro_f1: 0.4404
Epoch 5/20
8

## Mark hard examples

In [21]:
from utils.hard_samples_identify import get_hard_sample_indices

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probabilities = np.zeros((len(y_train), len(LABELS)))
for fold_train, fold_held_out in folds.split(X_train, y_train):
    fold_model = build_cnn()
    fold_model.fit(
        X_train[fold_train], keras.utils.to_categorical(y_train[fold_train], len(LABELS)),
        epochs=20, batch_size=32, verbose=0,
    )
    oof_probabilities[fold_held_out] = fold_model.predict(
        X_train[fold_held_out], batch_size=32, verbose=0
    )

val_probabilities = baseline_model.predict(X_val, batch_size=32, verbose=0)
test_probabilities = baseline_model.predict(X_test, batch_size=32, verbose=0)

# Each split gets its own percentile cutoff. This preserves the requested hard-sample
# proportion while preventing labels from influencing subset construction.
percentiles = [10, 20, 30, 40, 50]
hard_selection = {}
for percentile in percentiles:
    train_indices, train_cutoff = get_hard_sample_indices(oof_probabilities, percentile)
    val_indices, val_cutoff = get_hard_sample_indices(val_probabilities, percentile)
    test_indices, test_cutoff = get_hard_sample_indices(test_probabilities, percentile)
    hard_selection[percentile] = {
        "train_indices": train_indices,
        "val_indices": val_indices,
        "test_indices": test_indices,
        "train_cutoff": train_cutoff,
        "val_cutoff": val_cutoff,
        "test_cutoff": test_cutoff,
    }
    print(
        f"P{percentile}: train={len(train_indices)} (cutoff={train_cutoff:.4f}), "
        f"val={len(val_indices)} (cutoff={val_cutoff:.4f}), "
        f"test={len(test_indices)} (cutoff={test_cutoff:.4f})"
    )


P10: train=291 (cutoff=0.4559), val=97 (cutoff=0.2262), test=97 (cutoff=0.2139)
P20: train=581 (cutoff=0.7712), val=194 (cutoff=0.4531), test=194 (cutoff=0.4275)
P30: train=870 (cutoff=0.9128), val=290 (cutoff=0.5856), test=291 (cutoff=0.5997)
P40: train=1160 (cutoff=0.9606), val=387 (cutoff=0.7146), test=387 (cutoff=0.7064)
P50: train=1450 (cutoff=0.9849), val=484 (cutoff=0.8023), test=484 (cutoff=0.7864)


## Sweep targeted and random-control models


In [22]:
def train_weighted_model(hard_indices, val_indices, seed):
    keras.utils.set_random_seed(seed)
    model = keras.models.clone_model(baseline_model)
    model.set_weights(baseline_model.get_weights())
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    weights = np.ones(len(y_train))
    weights[hard_indices] = 1.5
    val_weights = np.ones(len(y_val))
    val_weights[val_indices] = 1.5
    model.fit(
        X_train, keras.utils.to_categorical(y_train, len(LABELS)),
        sample_weight=weights,
        validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS)), val_weights),
        epochs=10, batch_size=32,
        callbacks=[MacroF1Callback(X_val, y_val), keras.callbacks.EarlyStopping(
            monitor="val_macro_f1", mode="max", patience=1, restore_best_weights=True
        )],
        verbose=0,
    )
    return model

def train_random_control(number_hard, val_indices, seed):
    keras.utils.set_random_seed(seed)
    model = keras.models.clone_model(baseline_model)
    model.set_weights(baseline_model.get_weights())
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    random_indices = np.random.default_rng(seed).choice(len(y_train), size=number_hard, replace=False)
    weights = np.ones(len(y_train))
    weights[random_indices] = 1.5
    val_weights = np.ones(len(y_val))
    val_weights[val_indices] = 1.5
    model.fit(
        X_train, keras.utils.to_categorical(y_train, len(LABELS)),
        sample_weight=weights,
        validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS)), val_weights),
        epochs=10, batch_size=32,
        callbacks=[MacroF1Callback(X_val, y_val), keras.callbacks.EarlyStopping(
            monitor="val_macro_f1", mode="max", patience=1, restore_best_weights=True
        )],
        verbose=0,
    )
    return model


## Compare percentile results


In [23]:
def score_model(model, X, y, hard_indices):
    probabilities = model.predict(X, batch_size=32, verbose=0)
    predictions = probabilities.argmax(axis=1)
    hard_y = y[hard_indices]
    hard_predictions = predictions[hard_indices]
    return {
        "accuracy": accuracy_score(y, predictions),
        "macro_f1": f1_score(y, predictions, average="macro"),
        "hard_accuracy": accuracy_score(hard_y, hard_predictions),
        "hard_macro_f1": f1_score(hard_y, hard_predictions, average="macro"),
    }

baseline_overall = score_model(baseline_model, X_test, y_test, hard_selection[20]["test_indices"])
results = []
for percentile in percentiles:
    selection = hard_selection[percentile]
    baseline_scores = score_model(baseline_model, X_test, y_test, selection["test_indices"])
    results.append({
        "percentile": percentile,
        "strategy": "baseline",
        "train_hard_count": len(selection["train_indices"]),
        "test_hard_count": len(selection["test_indices"]),
        "train_cutoff": selection["train_cutoff"],
        "test_cutoff": selection["test_cutoff"],
        **baseline_scores,
        "delta_accuracy_vs_baseline": 0.0,
        "delta_macro_f1_vs_baseline": 0.0,
        "delta_hard_macro_f1_vs_baseline": 0.0,
    })
    for strategy, model_seed in [("targeted", 100 + percentile), ("random_control", 200 + percentile)]:
        if strategy == "targeted":
            model = train_weighted_model(
                selection["train_indices"], selection["val_indices"], seed=model_seed
            )
        else:
            model = train_random_control(
                len(selection["train_indices"]), selection["val_indices"], seed=model_seed
            )
        scores = score_model(model, X_test, y_test, selection["test_indices"])
        results.append({
            "percentile": percentile,
            "strategy": strategy,
            "train_hard_count": len(selection["train_indices"]),
            "test_hard_count": len(selection["test_indices"]),
            "train_cutoff": selection["train_cutoff"],
            "test_cutoff": selection["test_cutoff"],
            **scores,
            "delta_accuracy_vs_baseline": scores["accuracy"] - baseline_scores["accuracy"],
            "delta_macro_f1_vs_baseline": scores["macro_f1"] - baseline_scores["macro_f1"],
            "delta_hard_macro_f1_vs_baseline": scores["hard_macro_f1"] - baseline_scores["hard_macro_f1"],
        })

results_df = pd.DataFrame(results).sort_values(["percentile", "strategy"])
print("Baseline overall accuracy:", baseline_overall["accuracy"])
print("Baseline overall macro F1:", baseline_overall["macro_f1"])
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))


 - val_macro_f1: 0.7402
 - val_macro_f1: 0.7467
 - val_macro_f1: 0.7481
 - val_macro_f1: 0.7486
 - val_macro_f1: 0.7482
 - val_macro_f1: 0.7548
 - val_macro_f1: 0.7502
 - val_macro_f1: 0.7334
 - val_macro_f1: 0.7469
 - val_macro_f1: 0.7449
 - val_macro_f1: 0.7423
 - val_macro_f1: 0.7406
 - val_macro_f1: 0.7447
 - val_macro_f1: 0.7545
 - val_macro_f1: 0.7497
 - val_macro_f1: 0.7462
 - val_macro_f1: 0.7497
 - val_macro_f1: 0.7465
 - val_macro_f1: 0.7477
 - val_macro_f1: 0.7499
 - val_macro_f1: 0.7462
 - val_macro_f1: 0.7477
 - val_macro_f1: 0.7532
 - val_macro_f1: 0.7566
 - val_macro_f1: 0.7487
 - val_macro_f1: 0.7458
 - val_macro_f1: 0.7510
 - val_macro_f1: 0.7488
 - val_macro_f1: 0.7417
 - val_macro_f1: 0.7397
Baseline overall accuracy: 0.8057851239669421
Baseline overall macro F1: 0.7734486440009541
 percentile       strategy  train_hard_count  test_hard_count  train_cutoff  test_cutoff  accuracy  macro_f1  hard_accuracy  hard_macro_f1  delta_accuracy_vs_baseline  delta_macro_f1_vs_ba

## Save percentile results


In [ ]:
# os.makedirs("../models", exist_ok=True)
# results_df.to_csv("../models/experiment_7_percentile_results.csv", index=False)
# with open("../models/experiment_7_finbert_tokenizer.pkl", "wb") as f:
#     pickle.dump(tokenizer, f)
